In [1]:
import matplotlib.dates as mdates
import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
import scipy.ndimage
import detectRadioburst_OSRA as drb

# Set file name and other initializations as before
fname = "/net/lyot/scratch3/vocks/OSRA/2003/CD_300/031027_300.roh"

(dyspec,t_fits,f_fits,hdu)  = drb.read_osraf2(fname)
#(dyspec,f_fits) =  drb.cut_low(dyspec,f_fits,f_low_cut_val=30)
# Define the frequency array for f2
f2 = 400.0 - np.array(range(256)) * 200.0 / 255.0
f_fits = f2

# Simulate file size and stats
file_stats = os.stat(fname)
a1 = int(file_stats.st_size / 1040 + 0.5)

# Create a dummy start time
dummy_start = np.datetime64('2025-02-01T15:23:15.00')
t_fits = dummy_start + np.linspace(0, 1, a1).astype('timedelta64[D]')

# Initialize the spectrum array for f2 range only
dyspec = np.zeros((a1, 256), dtype=np.ubyte)

# Simulate reading the file
file = open(fname, "rb")
for i in range(a1):
    data_chunk = file.read(1040)
    np_data_chunk = np.frombuffer(data_chunk, dtype=np.uint8)
    year = int(np_data_chunk[0] / 16) * 10 + (np_data_chunk[0] & 15)
    if year > 50:
        year = year + 1900
    else:
        year = year + 2000

    # Extract time information
    month = int(np_data_chunk[1] / 16) * 10 + (np_data_chunk[1] & 15)
    day = int(np_data_chunk[2] / 16) * 10 + (np_data_chunk[2] & 15)
    hour = int(np_data_chunk[3] / 16) * 10 + (np_data_chunk[3] & 15)
    minute = int(np_data_chunk[4] / 16) * 10 + (np_data_chunk[4] & 15)
    second = int(np_data_chunk[5] / 16) * 10 + (np_data_chunk[5] & 15)
    microsecond = 100000 * np_data_chunk[6]

    t_fits[i] = datetime.datetime(year, month, day, hour, minute, second, microsecond)
    # Only take the relevant part of the data chunk for f2
    dyspec[i, :] = np_data_chunk[272: 528]  # Adjust the slice to match the f2 range

file.close()

# Normalize the data
dyspec_norm = np.zeros((a1, 256), dtype=np.ubyte)
for j in range(256):
    Imax = np.amax(dyspec[:, j])
    Imin = np.amin(dyspec[:, j])
    dyspec_norm[:, j] = (dyspec[:, j] - Imin) / (Imax - Imin) * 255

# Transpose the normalized data
dyspec_norm = dyspec_norm.T

# Define new frequency range (400 MHz to 200 MHz)
new_frequencies = np.linspace(400, 200, 1000)
target_steps = 5000

# Define the start and end time for the entire dataset
start_time = np.datetime64('2003-10-27T08:02:00.000')
end_time = np.datetime64('2003-10-27T08:12:00.000')

# Iterate over each hour
current_time = start_time
while current_time < end_time:
    next_time = current_time + np.timedelta64(10, 'm')

    # Create a boolean mask for the desired time range
    time_mask = (t_fits >= current_time) & (t_fits < next_time)

    # Extract the cutout arrays
    t_fits_cutout = t_fits[time_mask]
    dyspec_cutout = dyspec[time_mask, :]
    a2 = len(t_fits_cutout)

    if a2 == 0:
        current_time = next_time
        continue

    # Initialize resampled data for the cutout
    target_steps = 5000
    new_dyspec = np.zeros((target_steps, len(f_fits)), dtype=np.uint8)

    # Resample the frequency segment using scipy.ndimage.zoom
    zoomed_array = scipy.ndimage.zoom(dyspec_cutout, (target_steps/a2, 1))
    new_dyspec[:, :] = zoomed_array

    # Normalize new data
    new_dyspec_norm = ((new_dyspec - new_dyspec.min(axis=0)) /
                       (new_dyspec.ptp(axis=0) + 1e-6) * 255).astype(np.uint8)

    # Convert cutout time for plotting
    t_plot_cutout = mdates.date2num(t_fits_cutout)

    # Plot the resampled cutout
    plt.figure(figsize=(12, 6))
    plot_array = new_dyspec_norm.T
    plot_array = np.flip(plot_array, axis=0)
    im = plt.imshow(
        plot_array, aspect='auto', cmap='plasma',
        extent=[t_plot_cutout[0], t_plot_cutout[-1], f_fits[0], f_fits[-1]]
    )
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.setp(plt.gca().get_xticklabels(), rotation=45, ha='right')
    plt.xlabel('Time')
    plt.ylabel('Frequency (MHz)')
    plt.title(f'Resampled Frequency vs Time ({current_time.astype("datetime64[h]")} - {next_time.astype("datetime64[h]")})', fontweight="bold")
    plt.colorbar(im, label='Intensity')
    plt.yticks(np.linspace(f_fits[0], f_fits[-1], 10))
    plt.tight_layout()
    plt.show()

    current_time = next_time

(data_fits_new,data_fits_new_smooth) = drb.preproc(dyspec,gauss_sigma=1.5)

fig = plt.figure(figsize=(6, 4))
ax = plt.gca()

data_safe_arr = data_fits_new_smooth.ravel()
data_safe = np.sort(data_safe_arr)[int(data_safe_arr.shape[0] * 0.02):int(data_safe_arr.shape[0] * 0.98)]
vmin,vmax = [(np.nanmean(data_safe) - 2 * np.nanstd(data_safe)),
                  (np.nanmean(data_safe) + 4 * np.nanstd(data_safe)+0.5*np.nanmax(data_safe))]
        
ax.imshow(data_fits_new_smooth.T,aspect='auto',  origin='lower', vmax=vmax,vmin=vmin,
                   extent=[t_fits[0],t_fits[-1],f_fits[0],f_fits[-1]],cmap='inferno')

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title("OSRA Dynamic Spectrum")


print(dyspec_cutout.shape)

# background removal + Gaussian smoothing
(data_fits_new, data_fits_new_smooth) = drb.preproc(dyspec_cutout, gauss_sigma=1.5)


# binarization
bmap = drb.binarization(data_fits_new_smooth,N_order=6,peak_r=1.002)


fig,ax = plt.subplots(1,1,figsize=[6,3])
ax.imshow(1-bmap.T,aspect='auto', origin='lower',cmap='gray')



fig = plt.figure(figsize=[6,3])
plt.imshow(data_fits_new_smooth.T,aspect='auto',origin='lower',vmax=vmax,vmin=vmin/2-vmax/2)
plt.contour(bmap.T,[0,0.5,1],colors='k')
plt.title("overplot binary map to spectrum")


# detect verticle features
lines = drb.hough_detect(bmap,dyspec,threshold=40,line_gap=10,line_length=35,
            theta=np.linspace(np.pi/2-np.pi/8,np.pi/2-1/180*np.pi,300))

fig,ax = plt.subplots(1,1,figsize=[6,3])
lines = sorted(lines, key=lambda i: i[0][1])
ax.imshow(1-bmap.T,aspect='auto',origin='lower',cmap='gray')
for line in lines:
    p0,p1= line
    ax.plot( (p0[1], p1[1]),(p0[0], p1[0]),':')

line_sets = drb.line_grouping(lines)
# group the detected lines into group in regard of events


fig,ax = plt.subplots(1,1,figsize=[6,3])
ax.imshow(data_fits_new_smooth.T,aspect='auto',origin='lower',vmax=vmax,vmin=vmin,cmap='gray')

for idx,lines in enumerate(line_sets):
    for line in lines:
        p0,p1= line
        ax.plot( (p0[1], p1[1]),(p0[0], p1[0]),color='C'+str(idx+1))
        ax.plot( (p0[1], p1[1]),(p0[0], p1[0]),'k+',zorder=10)
    #ax.set_xlim((500,600))
#ax.set_ylim((bmap.shape[0], 0))


from type3detect import radioTools as rt

#(rt.freq_to_R(20e6)-rt.freq_to_R(80e6))/rt.c_r

(v_beam, f_range_burst, t_range_burst, model_curve_set,
     t_set_arr_set,f_set_arr_set,t_model_arr,f_model_arr
    )= drb.get_info_from_linegroup(line_sets,t_fits,f_fits)



plt.plot(t_model_arr,f_model_arr)

len(line_sets)

fig,ax = plt.subplots(1,1,figsize=[6,3],dpi=120)
lines = sorted(lines, key=lambda i: i[0][1])
ax.imshow(data_fits_new.T,aspect='auto',origin='lower', vmax=vmax,vmin=vmin,cmap='gray',
                   extent=[t_fits[0],t_fits[-1],f_fits[0],f_fits[-1]])
for idx,model in enumerate(model_curve_set):
    if (v_beam[idx] > 0) and (v_beam[idx] < 0.9):
        plt.plot(model[0],model[1],ls='--')
        plt.plot(t_range_burst[idx],f_range_burst[idx],'k+')
    


ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(hdu[0].header['CONTENT'])


#ax.set_xlim([t_fits[200],t_fits[400]])
#plt.ylim([10,88])


ValueError: not enough values to unpack (expected 4, got 3)